In [1]:
import time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import matplotlib.pyplot as plt
from torchvision import datasets, transforms, utils
from torch.utils.data import DataLoader
%cd "C:/Users/mateo/Desktop/polymath-jr-drifting"
import importlib
import src.matryoshka as matryoshka_utils
matryoshka_utils = importlib.reload(matryoshka_utils)
from src.driftXpress import *
from src.eval import (
    evaluate_generation,
    plot_conditional_mnist_results,
    plot_loss_history,
    plot_real_generated_projections,
)
from src.training import (
    compose_objective,
    format_epoch_losses,
    gradient_report,
    per_loss_gradient_norms,
    set_requires_grad,
)
from src.matryoshka import (
    VAE,
    LatentGenerator,
    f,
    conditional_drift_loss,
    evaluate_matryoshka_model,
    plot_matryoshka_diagnostics,
    plot_matryoshka_reconstructions,
    pretrained_full_joint_training_loop,
)
torch.manual_seed(7)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using:", device)


C:\Users\mateo\Desktop\polymath-jr-drifting
Using: cuda


In [2]:
transform = transforms.ToTensor()

SELECTED_DIGITS = (0, 8, 2, 4, 7, 9)
DIGIT_TO_CLASS = {digit: index for index, digit in enumerate(SELECTED_DIGITS)}
CLASS_TO_DIGIT = np.asarray(SELECTED_DIGITS)

train_data = datasets.MNIST("data", train=True, download=True, transform=transform)
test_data = datasets.MNIST("data", train=False, download=True, transform=transform)

def keep_selected_digits(dataset):
    selected = torch.tensor(SELECTED_DIGITS, dtype=dataset.targets.dtype)
    mask = torch.isin(dataset.targets, selected)
    dataset.data = dataset.data[mask]
    raw_targets = dataset.targets[mask]
    dataset.targets = torch.tensor(
        [DIGIT_TO_CLASS[int(digit)] for digit in raw_targets],
        dtype=torch.long,
    )
    return dataset

train_data = keep_selected_digits(train_data)
test_data = keep_selected_digits(test_data)

BATCH_SIZE = 500

train_loader = DataLoader(
    train_data,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)
test_loader = DataLoader(
    test_data,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)

print("Selected digits:", SELECTED_DIGITS)
print("Train samples:", len(train_data))
print("Test samples:", len(test_data))


Selected digits: (0, 8, 2, 4, 7, 9)
Train samples: 35788
Test samples: 6005


In [3]:
NUM_CLASSES = 6
LATENT_DIM = 32
MATRYOSHKA_DIMS = (4, 8, 16, 32)
model = f(
    VAE(input_dim=784, latent_dim=LATENT_DIM, matryoshka_dims=MATRYOSHKA_DIMS),
    LatentGenerator(latent_dim=LATENT_DIM, num_classes=NUM_CLASSES, matryoshka_dims=MATRYOSHKA_DIMS),
    num_classes=NUM_CLASSES,
)
print(f"Total parameters: {sum(parameter.numel() for parameter in model.parameters())}")
print(f"Trainable parameters: {sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)}")


Total parameters: 357280
Trainable parameters: 357280


In [4]:
def training_loop(
    num_epochs,
    T,
    num_landmarks,
    lambda_kl,
    lambda_drift,
    lambda_var,
    lambda_cov,
    lambda_representation=1.0,
    loss_scale=1.0,
    vae_pretrain_epochs=50,
    generator_warmup_epochs=30,
    generator_drift_warmup_epochs=25,
    generator_lr=1e-3,
    generator_drift_lr=1e-4,
    joint_generator_lr=1e-4,
    joint_vae_lr=1e-5,
    raw_drift_weight=0.25,
    lr=1e-4,
):
    # Keep pretrained-full-joint.ipynb's call signature and return tuple.
    return pretrained_full_joint_training_loop(
        num_epochs, T, num_landmarks, lambda_kl, lambda_drift,
        lambda_var, lambda_cov,
        lambda_representation=lambda_representation,
        loss_scale=loss_scale,
        vae_pretrain_epochs=vae_pretrain_epochs,
        generator_warmup_epochs=generator_warmup_epochs,
        generator_drift_warmup_epochs=generator_drift_warmup_epochs,
        generator_lr=generator_lr,
        generator_drift_lr=generator_drift_lr,
        joint_generator_lr=joint_generator_lr,
        joint_vae_lr=joint_vae_lr,
        raw_drift_weight=raw_drift_weight,
        lr=lr,
        train_loader=train_loader,
        device=device,
        num_classes=NUM_CLASSES,
        latent_dim=LATENT_DIM,
        matryoshka_dims=MATRYOSHKA_DIMS,
    )


In [ ]:
# Pretraining and all warmups use nested prefix losses before the fully joint phase.
VAE_PRETRAIN_EPOCHS = 50
GENERATOR_WARMUP_EPOCHS = 30
NUM_EPOCHS = 200
NUM_LANDMARKS = 200
TEMPERATURE = 2
LAMBDA_KL = 1e-5
LAMBDA_DRIFT = 30.0
LAMBDA_VAR = 0.0
LAMBDA_COV = 0.0
LAMBDA_REPRESENTATION = 1.0
LOSS_SCALE = 1.0  # Keep at 1.0; it is an intentional global gradient-scale test.

(
    f_trained,
    losses,
    recon_losses,
    kl_losses,
    drift_losses,
    var_losses,
    covar_losses,
    average_V,
    evolution,
    pretrain_history,
    generator_warmup_history,
    generator_drift_history,
    drift_raw_losses,
) = training_loop(
    NUM_EPOCHS,
    TEMPERATURE,
    NUM_LANDMARKS,
    LAMBDA_KL,
    LAMBDA_DRIFT,
    LAMBDA_VAR,
    LAMBDA_COV,
    lambda_representation=LAMBDA_REPRESENTATION,
    loss_scale=LOSS_SCALE,
    vae_pretrain_epochs=VAE_PRETRAIN_EPOCHS,
    generator_warmup_epochs=GENERATOR_WARMUP_EPOCHS,
    generator_drift_warmup_epochs=25,
    generator_lr=1e-3,
    generator_drift_lr=1e-4,
    joint_generator_lr=1e-4,
    joint_vae_lr=1e-5,
    raw_drift_weight=0.25,
    lr=1e-4,
)


Pretraining the VAE part for 50 epochs (generator frozen)...
-------- VAE pretrain epoch 1/50 --------
total=0.206302
representation=0.206302    scaled=0.206302
recon=0.206260    scaled=0.206260
kl=4.194881    scaled=0.000042
var=1.384952    scaled=0.000000
covar=0.029789    scaled=0.000000
-------- VAE pretrain epoch 5/50 --------
total=0.070595
representation=0.070595    scaled=0.070595
recon=0.069875    scaled=0.069875
kl=71.990227    scaled=0.000720
var=0.706224    scaled=0.000000
covar=7.891071    scaled=0.000000
-------- VAE pretrain epoch 10/50 --------
total=0.064239
representation=0.064239    scaled=0.064239
recon=0.063547    scaled=0.063547
kl=69.206197    scaled=0.000692
var=0.434232    scaled=0.000000
covar=37.090229    scaled=0.000000
-------- VAE pretrain epoch 15/50 --------
total=0.059216
representation=0.059216    scaled=0.059216
recon=0.058684    scaled=0.058684
kl=53.200266    scaled=0.000532
var=0.452225    scaled=0.000000
covar=76.464559    scaled=0.000000
--------

In [ ]:
# Optional: inspect the actual weighted gradient paths before calling backward.
import src.matryoshka as matryoshka_utils
DRIFT_CACHE = matryoshka_utils.DRIFT_CACHE
f_trained.train()
images, class_ids = next(iter(train_loader))
x = images.to(device).flatten(start_dim=1)
class_ids = class_ids.to(device)
noise = torch.randn(x.shape[0], f_trained.latent_dim, device=device)
mu, logvar, z_pos = f_trained.get_latent(x)
x_recon = f_trained.decode(z_pos)
z_raw, x_neg = f_trained.generate(noise, class_ids)
z_neg = f_trained.encode_mu(x_neg)
L_recon = matryoshka_utils.matryoshka_reconstruction_loss(f_trained.vae, z_pos, x, MATRYOSHKA_DIMS)
L_drift_reencoded, _ = conditional_drift_loss(z_neg, class_ids, DRIFT_CACHE, TEMPERATURE, MATRYOSHKA_DIMS)
L_drift_raw, _ = conditional_drift_loss(z_raw, class_ids, DRIFT_CACHE, TEMPERATURE, MATRYOSHKA_DIMS)
L_drift = L_drift_reencoded + 0.25 * L_drift_raw
print(per_loss_gradient_norms(
    {"recon": L_recon, "drift": L_drift},
    f_trained,
    {"drift": LAMBDA_DRIFT},
    lambda_representation=LAMBDA_REPRESENTATION,
))


In [ ]:
joint_history = {
    "total": losses,
    "recon": recon_losses,
    "kl": kl_losses,
    "drift": drift_losses,
    "drift_raw": drift_raw_losses,
    "var": var_losses,
    "covar": covar_losses,
    "force": average_V,
}
fig, axes = plot_loss_history(joint_history)
plt.show()


In [ ]:
# Full-space metrics keep the same interface; prefix metrics expose the nested behavior.
post_eval = evaluate_matryoshka_model(
    f_trained,
    test_loader,
    device=device,
    dimensions=MATRYOSHKA_DIMS,
    n_samples=2048,
    n_classes=NUM_CLASSES,
    class_to_label=CLASS_TO_DIGIT,
    include_mmd=False,
    include_fid=True,
)

samples = post_eval["samples"]
real_latents = samples["real_latents"]
generated_latents = samples["generated_latents"]
real_pixels = samples["real_images"]
generated_pixels = samples["generated_images"]
real_class_labels = samples["real_class_ids"]
generated_class_labels = samples["generated_class_ids"]
real_labels = samples["real_labels"]
generated_labels = samples["generated_labels"]
metrics = post_eval["metrics"]
latent_stats = post_eval["statistics"]

print("Latent-space metrics:")
for name, value in metrics.items():
    print(f"{name}: {value:.6f}")


In [ ]:
pixel_metrics = evaluate_generation(
    real_pixels,
    generated_pixels,
    generated_labels=generated_class_labels,
    n_classes=NUM_CLASSES,
    include_mmd=False,
)
print("Pixel-space metrics:")
for name, value in pixel_metrics.items():
    print(f"{name}: {value:.6f}")


In [ ]:
print("Latent statistics:")
for name, value in latent_stats.items():
    print(f"{name}: {value:.6f}")


In [ ]:
matryoshka_metrics = post_eval["matryoshka_metrics"]
matryoshka_statistics = post_eval["matryoshka_statistics"]
prefix_reconstruction_mse = post_eval["matryoshka_reconstruction_mse"]
print("Matryoshka prefix metrics:")
for dimension in MATRYOSHKA_DIMS:
    print(f"dimension={dimension}")
    for name, value in matryoshka_metrics[dimension].items():
        print(f"{name}: {value:.6f}")
    print(f"reconstruction_mse: {prefix_reconstruction_mse[dimension]:.6f}")

fig, axes = plot_matryoshka_diagnostics(
    matryoshka_metrics,
    prefix_reconstruction_mse,
)
plt.show()


In [ ]:
fig, axes = plot_conditional_mnist_results(
    f_trained,
    test_loader,
    device=device,
    samples=10,
    use_mean_for_reconstruction=True,
)
plt.show()

fig, axes = plot_matryoshka_reconstructions(
    f_trained,
    test_loader,
    MATRYOSHKA_DIMS,
    device=device,
    samples=10,
)
plt.show()


In [ ]:
projection_figures = plot_real_generated_projections(
    real_latents,
    generated_latents,
    real_labels=real_labels,
    generated_labels=generated_labels,
    methods=("pca", "umap"),
)
plt.show()


The prefix metric table, sweep plot, and reconstruction grid show whether the first 4/8/16 coordinates remain useful during fully joint training.